# 02 - Analisis exploratorio del dataset

Tarea 3.5 del WBS: validacion de la calidad del dataset y analisis exploratorio.
Las figuras y las tablas de este notebook alimentan directamente la memoria del
trabajo final y el dataset card.

Cada figura responde una pregunta concreta sobre si el dataset es apto para
entrenar, no es decorativa.

## 1. Entorno y datos

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
from pathlib import Path

from chessdl.config import load_config
from chessdl.data import schema

cfg = load_config()

# Bajar el dataset desde Hugging Face. Para analizar shards locales, reemplazar
# por: directorio = cfg.output.local_dir
from chessdl import hf
directorio = hf.download_dataset(cfg.output.hf_repo_id, "/content/ceia-chess/hub")

rutas = schema.shard_paths(directorio)
if not rutas:
    # Una version del paquete anterior al arreglo de `shard_paths` busca solo en
    # el primer nivel, y una descarga del Hub deja los shards bajo `data/`.
    rutas = sorted(
        ruta for ruta in Path(directorio).rglob("*.parquet")
        if not any(parte.startswith(".") for parte in ruta.relative_to(directorio).parts)
    )

tabla = schema.read_dataset(rutas)
print(f"{len(rutas)} shards, {tabla.num_rows:,} posiciones")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from chessdl import viz

viz.apply_style()
df = tabla.to_pandas()
df.head()

## 2. Chequeos de integridad

Los mismos chequeos del requerimiento 3.2 que corre el pipeline. Se repiten aca
porque el analisis exploratorio no tiene sentido sobre un dataset que no cierra.

In [ ]:
from chessdl.data.validate import validate_table

reporte = validate_table(tabla, cfg)
print(reporte.summary())

## 3. Cifras principales

Numeros sueltos: van como tabla, no como grafico. Un grafico de dos barras para
mostrar el balance de color diria menos que estas dos lineas.

In [ ]:
from chessdl.data.validate import describe_table

stats = describe_table(tabla)

filas = [
    ("Posiciones",              f"{stats['n_positions']:,}"),
    ("Partidas distintas",      f"{stats['n_games']:,}"),
    ("Posiciones por partida",  f"{stats['n_positions'] / stats['n_games']:.2f}"),
    ("Mueven blancas",          f"{stats['white_to_move_share']:.2%}"),
    ("Mueven negras",           f"{1 - stats['white_to_move_share']:.2%}"),
    ("Posiciones con mate",     f"{stats['mate_share']:.3%}"),
    ("value_white promedio",    f"{stats['value_mean']:+.4f}"),
    ("value_white desvio",      f"{stats['value_std']:.4f}"),
    ("ELO blancas promedio",    f"{stats['mean_white_elo']:,.0f}"),
    ("ELO negras promedio",     f"{stats['mean_black_elo']:,.0f}"),
    ("Ply promedio",            f"{stats['mean_ply']:.1f}"),
]
for etiqueta, valor in filas:
    print(f"{etiqueta:<24} {valor:>12}")

El balance de color debe dar practicamente 50/50: el muestreo lo garantiza por
construccion, tomando 2 posiciones con blancas al turno y 2 con negras en cada
partida (requerimiento 1.3).

`value_white` promedio deberia quedar levemente positivo: las blancas tienen la
ventaja de la primera jugada.

## 4. Distribucion de la etiqueta

La pregunta: **la etiqueta cubre todo el rango o se amontona en un lado?** Un
dataset concentrado cerca de cero le ensenaria a la red a predecir siempre
"equilibrado".

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["value_white"], bins=61, range=(-1, 1),
        color=viz.SEQUENTIAL, edgecolor=viz.SURFACE, linewidth=0.5)
ax.axvline(0, color=viz.INK_MUTED, linewidth=1, linestyle=":")
viz.label_axes(
    ax,
    "Distribucion de la evaluacion normalizada",
    "value_white   (-1 ganan negras   ·   +1 ganan blancas)",
    "posiciones",
    note="Los picos en los extremos son mates y ventajas decisivas recortadas a +-1",
)
fig.tight_layout()

## 5. Los dos puntos de vista de la etiqueta

La red se entrena sobre `value_stm`, la etiqueta en perspectiva del jugador al
turno, con el tablero espejado cuando mueven las negras. Esta seccion mira que
relacion hay entre las dos mitades del dataset.

**Estas dos distribuciones no tienen por que coincidir, y de hecho no deben.**
El ajedrez no es simetrico: las blancas mueven primero y puntuan alrededor del
53-55%, asi que `value_white` tiene media positiva. Como

    value_stm = +value_white  con blancas al turno
    value_stm = -value_white  con negras al turno

la mitad con blancas al turno queda corrida hacia valores positivos y la mitad
con negras al turno, hacia negativos. Ese corrimiento **es** la ventaja de la
primera jugada, medida sobre el dataset.

Lo que valida el espejado del tablero no es este grafico sino la seccion 10,
donde una posicion y su espejo con los colores invertidos codifican al mismo
tensor. Eso es una igualdad exacta; un chequeo estadistico nunca podria
distinguir un error de codigo de una propiedad del juego.

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(-1, 1, 61)
ax.hist(df.loc[df["turn_white"], "value_stm"], bins=bins, histtype="step",
        linewidth=2, color=viz.SERIES[0], label="mueven blancas")
ax.hist(df.loc[~df["turn_white"], "value_stm"], bins=bins, histtype="step",
        linewidth=2, linestyle="--", color=viz.SERIES[1], label="mueven negras")
ax.axvline(0, color=viz.INK_MUTED, linewidth=1, linestyle=":")
ax.legend(loc="upper center")
viz.label_axes(
    ax,
    "La ventaja de las blancas, vista desde el jugador al turno",
    "value_stm   (+1 gana quien mueve)",
    "posiciones",
    note="El corrimiento entre las curvas es la ventaja de mover primero, no un error",
)
fig.tight_layout()

In [ ]:
# El control con sentido: reflejar la mitad de las negras las pone en la misma
# escala que las blancas. Ahora si tienen que superponerse.
blancas = df.loc[df["turn_white"], "value_stm"]
negras_reflejada = -df.loc[~df["turn_white"], "value_stm"]

fig, ax = plt.subplots()
ax.hist(blancas, bins=bins, histtype="step", linewidth=2,
        color=viz.SERIES[0], label="mueven blancas")
ax.hist(negras_reflejada, bins=bins, histtype="step", linewidth=2, linestyle="--",
        color=viz.SERIES[1], label="mueven negras (reflejada)")
ax.legend(loc="upper center")
viz.label_axes(
    ax,
    "Consistencia entre las dos mitades del dataset",
    "value_white equivalente",
    "posiciones",
    note="Reflejada, la mitad de las negras debe superponerse con la de las blancas",
)
fig.tight_layout()

In [ ]:
from chessdl.normalize import value_to_cp

print(f"{'':<26}{'blancas al turno':>18}{'negras al turno':>18}")
print("-" * 62)
for etiqueta, f in [("media", np.mean), ("desvio", np.std),
                    ("mediana", np.median), ("|valor| medio", lambda x: np.abs(x).mean())]:
    print(f"{etiqueta:<26}{f(blancas):>18.4f}{f(negras_reflejada):>18.4f}")

print()
print(f"Diferencia de medias tras reflejar : {blancas.mean() - negras_reflejada.mean():+.4f}")
print("  (deberia ser chica: mide si las dos mitades son comparables)")
print()
print(f"Ventaja de las blancas en el dataset: {df['value_white'].mean():+.4f}")
inversa = value_to_cp(df["value_white"].mean(), cfg.normalization.scale, cfg.normalization.cp_clip)
print(f"  equivale a {inversa:+.1f} centipeones")
print("  (este es el numero que va a la memoria)")

## 6. De donde salen las posiciones

La pregunta: **el muestreo cubre la partida entera?** El plan decide
explicitamente **no** saltear aperturas, asi que tiene que haber posiciones
tempranas.

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["ply"], bins=60, color=viz.SEQUENTIAL, edgecolor=viz.SURFACE, linewidth=0.5)
viz.label_axes(
    ax,
    "De que momento de la partida salen las posiciones",
    "ply (media jugada dentro de la partida)",
    "posiciones",
    note="Sin corte en la apertura: el muestreo cubre la partida completa",
)
fig.tight_layout()

In [ ]:
print(f"Ply minimo   : {df['ply'].min()}")
print(f"Ply maximo   : {df['ply'].max()}")
print(f"Ply mediano  : {df['ply'].median():.0f}")
print(f"Posiciones en las primeras 10 medias jugadas: "
      f"{(df['ply'] < 10).mean():.2%}")

## 7. Como se definen las partidas

La pregunta: **el dataset contiene posiciones de todas las fases?** Se espera que
la ventaja promedio (en valor absoluto) crezca con el avance de la partida: las
aperturas estan equilibradas, los finales estan definidos.

In [ ]:
cortes = np.arange(0, int(df["ply"].max()) + 11, 10)
indice = np.digitize(df["ply"], cortes) - 1

centros, medias, conteos = [], [], []
for b in range(len(cortes) - 1):
    seleccion = df["value_white"].to_numpy()[indice == b]
    if len(seleccion) >= 30:   # ignorar buckets con muy pocos datos
        centros.append((cortes[b] + cortes[b + 1]) / 2)
        medias.append(np.abs(seleccion).mean())
        conteos.append(len(seleccion))

fig, ax = plt.subplots()
ax.plot(centros, medias, color=viz.SERIES[0], marker="o")
ax.set_ylim(0, 1)
viz.label_axes(
    ax,
    "Cuan definidas estan las posiciones segun avanza la partida",
    "ply",
    "|value_white| promedio",
    note="Mas alto significa una ventaja mas clara para alguno de los dos bandos",
)
fig.tight_layout()

## 8. Calidad de las partidas de origen

Verificacion visual del filtro: ninguna barra puede quedar a la izquierda del
umbral de ELO configurado (requerimiento 1.1).

In [ ]:
fig, ax = plt.subplots()
elos = np.concatenate([df["white_elo"].to_numpy(), df["black_elo"].to_numpy()])
ax.hist(elos, bins=40, color=viz.SEQUENTIAL, edgecolor=viz.SURFACE, linewidth=0.5)
ax.axvline(cfg.filter.min_elo, color=viz.INK_SECONDARY, linewidth=1.5, linestyle="--")
ax.text(cfg.filter.min_elo, ax.get_ylim()[1] * 0.96,
        f"  umbral {cfg.filter.min_elo}", color=viz.INK_SECONDARY, fontsize=8, va="top")
viz.label_axes(
    ax,
    "Distribucion de ELO de los jugadores",
    "ELO",
    "jugadores (los dos colores juntos)",
    note="Nada puede quedar a la izquierda de la linea de umbral",
)
fig.tight_layout()

## 9. Composicion por control de tiempo

In [ ]:
composicion = df["time_control"].value_counts()
for categoria, cantidad in composicion.items():
    print(f"{categoria:<16}{cantidad:>10,}   {cantidad / len(df):>7.2%}")

print()
excluidos = set(composicion.index) - set(cfg.filter.time_controls)
print("Categorias inesperadas:", excluidos or "ninguna")

## 10. Codificación de la posición como tensor

Esta sección verifica la decisión de diseño central del proyecto: **la red ve
siempre el tablero desde la perspectiva del jugador que mueve**.

El dataset guarda FENs, no tensores — el etiquetado con Stockfish es la parte
cara e irrepetible, mientras que la codificación es barata y puede cambiar sin
re-etiquetar nada. El tensor se calcula al vuelo, y es esta función la que
consumirá el `Dataset` de PyTorch en el bloque 4 del WBS.

In [ ]:
import chess
from chessdl.encoding import (
    N_PLANES, TENSOR_SHAPE, PLANE_EN_PASSANT, PLANE_HALFMOVE_CLOCK,
    PLANE_OWN_CASTLE_KINGSIDE, PLANE_OWN_PIECES, PLANE_OPPONENT_PIECES,
    board_to_tensor, to_stm_perspective,
)

# Una posicion real del dataset, con las negras al turno.
fila = df[~df["turn_white"]].iloc[0]
tablero = chess.Board(fila["fen"])

tensor = board_to_tensor(tablero)
print("FEN      :", fila["fen"])
print("Al turno :", "blancas" if tablero.turn else "negras")
print("Tensor   :", tensor.shape, tensor.dtype)
print("Planos   :", N_PLANES)

### El espejado

Cuando mueven las negras el tablero se espeja y se intercambian los colores, de
modo que las piezas propias caen siempre en los planos 0–5. Abajo, el tablero
original y el que efectivamente ve la red.

In [ ]:
print("Tablero original (mueven negras):")
print(tablero)
print()
print("Lo que ve la red (siempre 'blancas al turno'):")
print(to_stm_perspective(tablero))

### La verificación que importa

Una posición y su espejo con los colores invertidos son **estratégicamente la
misma situación**. Si el espejado está bien hecho, la red las tiene que ver
idénticas: "me toca mover y estoy mejor" es lo mismo sin importar de qué color
sean mis piezas.

Esto es lo que duplica el aprovechamiento efectivo de los datos: la red no tiene
que aprender dos representaciones simétricas del mismo concepto.

In [ ]:
espejada = tablero.mirror()

print("Son tableros distintos:", tablero.fen() != espejada.fen())
print("  original :", tablero.fen())
print("  espejado :", espejada.fen())
print()
print("Pero codifican al MISMO tensor:",
      np.array_equal(board_to_tensor(tablero), board_to_tensor(espejada)))

### Contenido de los planos

Los peones propios, tal como los ve la red. La fila 0 es la primera fila del
jugador que mueve, así que los peones propios aparecen siempre abajo,
independientemente del color real de las piezas.

In [ ]:
def mostrar_plano(plano, titulo):
    """Imprime un plano 8x8 con la fila 0 (propia) abajo, como un tablero."""
    print(titulo)
    for fila_idx in range(7, -1, -1):
        celdas = " ".join("#" if v else "." for v in plano[fila_idx])
        print(f"  {fila_idx}  {celdas}")
    print("     a b c d e f g h")

peones_propios = tensor[PLANE_OWN_PIECES + chess.PAWN - 1]
peones_rival = tensor[PLANE_OPPONENT_PIECES + chess.PAWN - 1]

mostrar_plano(peones_propios, "Plano 0 - peones propios")
print()
mostrar_plano(peones_rival, "Plano 6 - peones del rival")

In [ ]:
# Resumen de la ocupacion de todos los planos.
nombres = ["peon", "caballo", "alfil", "torre", "dama", "rey"]

print(f"{'plano':<7}{'contenido':<28}{'casillas activas':>18}")
print("-" * 53)
for i, nombre in enumerate(nombres):
    print(f"{i:<7}{'propio: ' + nombre:<28}{int(tensor[i].sum()):>18}")
for i, nombre in enumerate(nombres):
    print(f"{i + 6:<7}{'rival: ' + nombre:<28}{int(tensor[i + 6].sum()):>18}")
for i, etiqueta in enumerate(["enroque propio corto", "enroque propio largo",
                              "enroque rival corto", "enroque rival largo"]):
    print(f"{i + 12:<7}{etiqueta:<28}{int(tensor[i + 12].sum()):>18}")
print(f"{PLANE_EN_PASSANT:<7}{'captura al paso':<28}{int(tensor[PLANE_EN_PASSANT].sum()):>18}")
print(f"{PLANE_HALFMOVE_CLOCK:<7}{'reloj de 50 jugadas':<28}{tensor[PLANE_HALFMOVE_CLOCK][0][0]:>18.2f}")

### Consistencia sobre el dataset

El tensor se calcula desde el FEN guardado. Verificamos sobre una muestra que
todas las posiciones codifican correctamente y que las piezas propias caen
siempre donde deben.

In [ ]:
muestra = df.sample(min(500, len(df)), random_state=0)

formas_ok = 0
reyes_ok = 0
for fen in muestra["fen"]:
    t = board_to_tensor(chess.Board(fen))
    formas_ok += t.shape == TENSOR_SHAPE
    # Siempre tiene que haber exactamente un rey propio y uno del rival.
    rey_propio = t[PLANE_OWN_PIECES + chess.KING - 1].sum()
    rey_rival = t[PLANE_OPPONENT_PIECES + chess.KING - 1].sum()
    reyes_ok += (rey_propio == 1 and rey_rival == 1)

print(f"Posiciones verificadas    : {len(muestra):,}")
print(f"Forma correcta            : {formas_ok:,}")
print(f"Un rey propio y uno rival : {reyes_ok:,}")

## 11. Transformación inversa a centipeones

Requerimiento 4.2: las evaluaciones tienen que poder presentarse en la misma
escala que usa Stockfish, aplicando la inversa de la normalización del
requerimiento 1.1.

Es la función que va a usar la CLI del motor para reportar sus evaluaciones en
centipeones en vez de en el rango normalizado.

In [ ]:
from chessdl.normalize import cp_to_value, value_to_cp

escala = cfg.normalization.scale
recorte = cfg.normalization.cp_clip

print(f"value = tanh(cp / {escala:.0f})        cp = {escala:.0f} * atanh(value)")
print(f"Recorte de cp: +-{recorte}")
print()
print(f"{'cp original':>14}{'value':>12}{'cp recuperado':>16}")
print("-" * 42)
for cp in [-2000, -800, -300, -50, 0, 50, 300, 800, 2000]:
    v = cp_to_value(cp, escala, recorte)
    print(f"{cp:>14}{v:>12.4f}{value_to_cp(v, escala, recorte):>16.1f}")

In [ ]:
# Sobre el dataset real: recuperar los centipeones desde la etiqueta guardada.
recuperados = np.array([value_to_cp(v, escala, recorte) for v in df["value_white"]])
error = np.abs(recuperados - df["cp_white"].to_numpy())

print(f"Error maximo al reconstruir cp: {error.max():.4f} centipeones")
print(f"Error medio                   : {error.mean():.6f} centipeones")
print()
print("Una red que sature y prediga exactamente +-1 tampoco rompe la inversa:")
for v in [-1.0, 1.0]:
    print(f"  value_to_cp({v:+.1f}) = {value_to_cp(v, escala, recorte):+.1f} cp")

## 12. Conclusiones

Completar al ejecutar sobre el dataset definitivo. Los puntos a dejar asentados
en la memoria:

- volumen final (posiciones y partidas distintas);
- balance de color medido contra el 50% teorico;
- forma de la distribucion de la etiqueta y proporcion de posiciones recortadas
  al limite de +-1;
- diferencia entre las dos curvas de la seccion 5 (evidencia de que el espejado
  es correcto);
- cobertura de fases de la partida;
- version exacta de Stockfish y profundidad usadas, que salen de las columnas
  `sf_version` y `sf_depth`.

In [ ]:
print("Version del motor :", set(df["sf_version"].unique()))
print("Profundidad       :", set(df["sf_depth"].unique()))
print("Dumps de origen   :", set(df["src_dump"].unique()))